In [1]:
%pip install yfinance

# extract data from yfinance and return as a pandas dataframe

import yfinance as yf
import pandas as pd


def get_stock_data(ticker):
    df = yf.download(
        ticker,
        period="1y",
        auto_adjust=True,
        progress=False
    )

    if df.empty:
        raise ValueError(f"No data returned for {ticker}")

    # Flatten columns if yfinance returns MultiIndex
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df["ticker"] = ticker

    return df.reset_index()

StatementMeta(, ccf025fd-2ad3-42d7-86b1-3736ca9b919c, 8, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 92.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 182.8 MB/s  0:00:00
  Attempting uninstall: cffi0m╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/9 [lxml]e]
    Found existing installation: cffi 2.1.0━━━━━━━━━━━━━━━━━━━ 3/9 [lxml]
    Not uninstalling cffi at /home/trusted-service-user/cluster-env/trident_env/lib/python3.13/site-packages, outside environment /nfs4/pyenv-74e6f0f8-dcd9-4a99-a929-735094b5c808
    Can't uninstall 'cffi'. No files were found to uninstall.━ 3/9 [lxml]
  Attempting uninstall: certifi━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/9 [lxml]
    Found existing installation: certifi 2026.6.17━━━━━━━━━━━━ 3/9 [lxml]
    Not uninstalling certifi at /home/trusted-service-user/cluster-env/trident_env/lib/python3.13/site-packages, outside environment /nfs4/pyenv-74e6f0f8-dcd9-4a99-a929-735094b5c808
    Can't uninstall 'certifi'. No files were found to uninstall.0m 3/9 [lxml]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 

In [2]:
# Load stock prices from Yahoo Finance
# and write directly to the Lakehouse Bronze table

import pandas as pd

TICKERS = [
    "QQQM",
    "MSFT",
    "GOOGL",
    "AMZN",
    "SPCX"
]

all_data = []

for ticker in TICKERS:

    print(f"Downloading {ticker}...")

    try:
        df = get_stock_data(ticker)

        all_data.append(df)

        print(f"Success: {ticker}")

    except Exception as e:
        print(f"Failed: {ticker} - {e}")

if not all_data:
    raise Exception("No stock data was retrieved.")

combined_df = pd.concat(
    all_data,
    ignore_index=True
)

# Convert pandas dataframe to Spark dataframe
spark_df = spark.createDataFrame(combined_df)

# Write to Bronze Delta table in the Lakehouse
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_stock_prices")

print(f"Rows loaded: {spark_df.count()}")
print("Data written to bronze_stock_prices")

StatementMeta(, ccf025fd-2ad3-42d7-86b1-3736ca9b919c, 10, Finished, Available, Finished, False)

Success: QQQM
Success: MSFT
Success: GOOGL
Success: AMZN
Success: SPCX
Rows loaded: 1103
Data written to bronze_stock_prices


In [3]:
# Validate Silver Delta table load by displaying a sample of cleaned records


display(
    spark.sql("""
        SELECT *
        FROM bronze_stock_prices
        LIMIT 10
    """)
)

StatementMeta(, ccf025fd-2ad3-42d7-86b1-3736ca9b919c, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3a99ddee-03b9-4575-949c-126157cdc0dd)